In [48]:
from pathlib import Path
import os
import json
from pypdf import PdfReader
import pandas as pd
import sqlite3

In [22]:
home_path = Path("/home/daniel/code/exercises_classifier/")

In [23]:
with open(home_path / "topics.json", mode="r", encoding="utf-8") as file:
    topics_dict = json.load(file)

In [24]:
def extract_info_from_pdf(pdf_file):
    reader = PdfReader(pdf_file)
    text = reader.pages[0].extract_text()
    text_lines = [line.strip() for line in text.split("\n") if line.strip()][3:]
    
    return text_lines

In [12]:
text_lines

['2025',
 'QUÍMICA',
 'TEMA 6: EQUILIBRIOS ÁCIDO-BASE',
 '• Junio, Ejercicio 3B',
 '• Reserva 1, Ejercicio 2B',
 '• Reserva 1, Ejercicio 5',
 '• Reserva 2, Ejercicio 3B']

In [40]:
def parse_lines(text_lines):
    
    year = int(text_lines[0])
    subject = text_lines[1].lower()
    topic = text_lines[2].split(":")[1].strip().lower()
    topic = topics_dict[topic]

    for line in text_lines[3:]:
        line = line[2:]  # remove "• "
        components = line.split(', ')
        exam = components[0]
        exercise = ' '.join(components[1:])
            
    return f"{subject}|{topic}|{year}|{exam}|{exercise}" 

In [7]:
os.chdir("./pdf/química")
os.getcwd()

'/home/daniel/code/exercises_classifier/pdf/química'

In [21]:
Path.cwd()

PosixPath('/home/daniel/code/exercises_classifier/pdf/química')

In [25]:
pdf_files_path = home_path / "pdf/química"
folders = [folder for folder in pdf_files_path.iterdir() if folder.name != ".DS_Store"]
folders

[PosixPath('/home/daniel/code/exercises_classifier/pdf/química/ácido base'),
 PosixPath('/home/daniel/code/exercises_classifier/pdf/química/redox'),
 PosixPath('/home/daniel/code/exercises_classifier/pdf/química/equilibrio químico'),
 PosixPath('/home/daniel/code/exercises_classifier/pdf/química/solubilidad'),
 PosixPath('/home/daniel/code/exercises_classifier/pdf/química/reactividad orgánica'),
 PosixPath('/home/daniel/code/exercises_classifier/pdf/química/formulación'),
 PosixPath('/home/daniel/code/exercises_classifier/pdf/química/cantidad química'),
 PosixPath('/home/daniel/code/exercises_classifier/pdf/química/átomo'),
 PosixPath('/home/daniel/code/exercises_classifier/pdf/química/termoquímica'),
 PosixPath('/home/daniel/code/exercises_classifier/pdf/química/enlace químico')]

In [41]:
with open(home_path / "classifications.csv", mode="w", encoding="utf-8") as f:
    headers =  "asignatura,tema,año,convocatoria,ejercicio,tipo_ejercicio"
    f.write(headers + "\n")
    for folder in folders:
        for file in folder.iterdir():
            if file.suffix == ".pdf":
                try:
                    lines = extract_info_from_pdf(file)
                    information = parse_lines(lines)
                    f.write(information + ",\n")
                except Exception as e:
                    print(f"Error: {e}. Error parsing file: {file}")
                    continue

In [19]:
file = "/home/daniel/code/exercises_classifier/pdf/química/ácido base/2025 - Ácido Base.pdf"
lines = extract_info_from_pdf(file)
parse_lines(lines)

'química|ácido base|2025|Reserva 2|Ejercicio 3B'

In [36]:
os.chdir(home_path)

In [46]:
df = pd.read_csv("classifications.csv", sep="|")

In [47]:
df

,asignatura,tema,año,convocatoria,ejercicio,tipo_ejercicio
0,química,ácido base,2012,Septiembre,Ejercicio 5 Opción B,NaN
1,química,ácido base,2011,Septiembre,Ejercicio 4 Opción B,NaN
2,química,ácido base,2009,Septiembre,Ejercicio 4 Opción B,NaN
3,química,ácido base,2014,Septiembre,Ejercicio 5 Opción B,NaN
4,química,ácido base,2024,Julio,Ejercicio C3,NaN
...,...,...,...,...,...,...
228,química,enlace químico,2014,Septiembre,Ejercicio 2 Opción B,NaN
229,química,enlace químico,2012,Septiembre,Ejercicio 2 Opción B,NaN
230,química,enlace químico,2019,Septiembre,Ejercicio 2 Opción B,NaN
231,química,enlace químico,2022,Julio,Ejercicio B3,NaN


In [49]:
conn = sqlite3.connect(home_path / "database.db")

In [50]:
conn.execute("""
            CREATE TABLE IF NOT EXISTS ejercicios (
                id           INTEGER PRIMARY KEY AUTOINCREMENT,
                asignatura   TEXT    NOT NULL,
                tema         TEXT    NOT NULL,
                año          INTEGER NOT NULL,
                convocatoria TEXT    NOT NULL,
                ejercicio    TEXT    NOT NULL,
                tipo_ejercicio TEXT  NOT NULL
            )
        """)
conn.commit()

In [51]:
df.to_sql("ejercicios", conn, if_exists="replace", index = False)

233